# LangGraph + LangChain
* 15.1에서 만든 **Node · Graph**와 15.2의 **`add_messages` · Conditional Edge**에 LangChain LLM을 연결합니다.

---
* `day15` Conda 환경을 사용합니다.
* LangChain OpenAI 연동 패키지를 추가로 설치합니다.

In [1]:
!pip install langchain-openai

In [3]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

WORKDIR : /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/15일차


---
## Node 안에서 LangChain LLM 호출

Node 안에서 **`ChatOpenAI.invoke()`** 로 실제 LLM 응답을 받습니다.

| Message 타입 | 역할 |
|----------------|------|
| `HumanMessage` | 사용자 메시지 |
| `AIMessage` | 모델 응답 메시지 |
| `SystemMessage` | 시스템 프롬프트 (역할·규칙 지정) |

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=OPENAI_API_KEY)

# 그래프 밖에서 먼저 단독 호출해 봅니다
demo_reply = llm.invoke([
    SystemMessage(content='한 문장으로만 답하세요.'),
    HumanMessage(content='LangGraph가 뭐야?'),
])
demo_reply.content

'LangGraph는 자연어 처리 및 기계 학습을 위한 언어 모델을 구축하고 활용하는 그래프 기반 시스템입니다.'

* Node 내부에서 LangChain 기반 LLM을 실행해 보겠습니다

In [5]:
from pydantic import BaseModel

class ChatState(BaseModel):
    question: str = ''
    answer: str = ''

In [6]:
def chat_node(state: ChatState) -> dict:
    """LangChain LLM을 Node 안에서 1회 호출합니다."""
    response = llm.invoke([
        SystemMessage(content='친절한 한국어 어시스턴트입니다. 2문장 이내로 답하세요.'),
        HumanMessage(content=state.question),
    ])
    return {'answer': response.content}

In [7]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)
workflow.add_node('chat', chat_node)
workflow.add_edge(START, 'chat')
workflow.add_edge('chat', END)

single_app = workflow.compile()

In [8]:
result = single_app.invoke(ChatState(question='LangGraph의 Node는 무엇을 하나요?'))
print('질문:', result['question'])
print('답변:', result['answer'])

질문: LangGraph의 Node는 무엇을 하나요?
답변: LangGraph의 Node는 다양한 정보를 저장하고 처리하는 기본 단위로, 서로 연결되어 데이터를 표현하고 분석하는 역할을 합니다. 각 Node는 특정한 속성이나 기능을 가질 수 있어 복잡한 데이터 구조를 형성할 수 있습니다.


In [ ]:
for chunk in single_app.stream(ChatState(question='LangGraph에서 Edge는 무엇인가요?')):
    print(chunk)

{'chat': {'answer': 'LangGraph에서 Edge는 노드 간의 관계를 나타내는 연결선으로, 두 노드를 연결하여 그들 간의 상호작용이나 관계를 표현합니다. Edge는 방향성과 가중치를 가질 수 있어, 다양한 유형의 데이터 및 관계를 모델링하는 데 사용됩니다.'}}


---
## `chat_history`에 LangChain 메시지 저장

LangChain의 `HumanMessage` / `AIMessage`를 그대로 history에 쌓고,  
다음 턴에 **전체 history를 LLM에 넘겨** 맥락을 유지합니다.

In [10]:
from typing import Annotated

from pydantic import Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class ChatState(BaseModel):
    # 15.2와 동일: add_messages → Node는 새 메시지 리스트만 반환
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    question: str = ''  # 이번 턴에 추가할 사용자 질문

In [ ]:
def multi_chat_node(state: ChatState) -> dict:
    # 이번 턴 사용자 메시지를 history에 추가
    user_msg = HumanMessage(content=state.question) # -> HumanMessage: 사용자(인간)의 메시지를 저장하는 메시지 클래스
    messages_for_llm = state.chat_history + [user_msg] #state.chat_history안에 AI가 한 답변이랑 System prompt 정보, 내가 한 질문들도 포함되어 있음. 

    # 지금까지의 대화 전체를 LLM에 전달
    response = llm.invoke([
        SystemMessage(content='친절한 한국어 어시스턴트입니다. 이전 대화를 기억하세요.'),
        *messages_for_llm,
    ])

    # add_messages 덕분에 [user_msg, response]만 반환해도 기존 history에 이어 붙음
    return {'chat_history': [user_msg, response]}


multi_workflow = StateGraph(ChatState)
multi_workflow.add_node('chat', multi_chat_node)
multi_workflow.add_edge(START, 'chat')
multi_workflow.add_edge('chat', END)

multi_app = multi_workflow.compile()

## 이름 기억 테스트

In [12]:
state = multi_app.invoke(ChatState(question='내 이름은 민수야. 기억해줘.'))

print('--- 1턴 ---')
for msg in state['chat_history']:
    role = '사용자' if msg.type == 'human' else 'AI'
    print(f'[{role}] {msg.content}')

--- 1턴 ---
[사용자] 내 이름은 민수야. 기억해줘.
[AI] 안녕하세요, 민수님! 반갑습니다. 어떻게 도와드릴까요?


In [13]:
# 이전 state를 이어서 새 질문만 넣습니다
state = multi_app.invoke(ChatState(
    chat_history=state['chat_history'],
    question='내 이름이 뭐였지?',
))

print('--- 2턴 이후 전체 history ---')
for msg in state['chat_history']:
    role = '사용자' if msg.type == 'human' else 'AI'
    print(f'[{role}] {msg.content}')

--- 2턴 이후 전체 history ---
[사용자] 내 이름은 민수야. 기억해줘.
[AI] 안녕하세요, 민수님! 반갑습니다. 어떻게 도와드릴까요?
[사용자] 내 이름이 뭐였지?
[AI] 민수님이라고 하셨습니다! 다른 질문이나 도움이 필요하시면 언제든지 말씀해 주세요.


### `add_messages`가 필요한 이유

add_message 속성 추가 없이 Node가 **새 메시지만** 반환하면  
기존 history가 **통째로 덮어써져** 이전 대화가 사라집니다.

In [14]:
class BrokenChatState(BaseModel):
    chat_history: list[BaseMessage] = Field(default_factory=list) # -> Annotated[, add_messages] 없음
    question: str = ''


def broken_chat_node(state: BrokenChatState) -> dict:
    user_msg = HumanMessage(content=state.question)
    response = llm.invoke([HumanMessage(content=state.question)])
    return {'chat_history': [user_msg, response]}  # 기존 history 무시 → 덮어쓰기


broken_app = StateGraph(BrokenChatState)
broken_app.add_node('chat', broken_chat_node)
broken_app.add_edge(START, 'chat')
broken_app.add_edge('chat', END)
broken_app = broken_app.compile()

s1 = broken_app.invoke(BrokenChatState(question='내 이름은 영희야.'))
s2 = broken_app.invoke(BrokenChatState(
    chat_history=s1['chat_history'],
    question='내 이름이 뭐였지?',
))

print('history 메시지 수:', len(s2['chat_history']))
print('마지막 AI 답:', s2['chat_history'][-1].content)

history 메시지 수: 2
마지막 AI 답: 죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 제가 도와드릴 수 있는 다른 것이 있나요?


### `input()` 기반으로 채팅하기
* LangGraph로 완전한 멀티턴 대화를 구현하려면, 대화 지속/종료를 선택하는 분기가 필요합니다
* `add_conditional_edges`와 적절한 `router`를 활용해 `input()` 기반으로 채팅을 구현해 보세요

In [29]:
from typing import Annotated

from pydantic import Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class ChatState(BaseModel):
    # 15.2와 동일: add_messages → Node는 새 메시지 리스트만 반환
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    question: str = ''  # 이번 턴에 추가할 사용자 질문

def chat_node(state: ChatState) -> dict:
    # 이번 턴 사용자 메시지를 history에 추가
    user_msg = HumanMessage(content=state.question)
    # 이전 턴 history + 이번 질문을 모두 LLM에 전달 (이름 기억 등 맥락 유지)
    messages_for_llm = list(state.chat_history) + [user_msg]

    response = llm.invoke([
        SystemMessage(content='친절한 한국어 어시스턴트입니다. 이전 대화를 기억하고 이어서 답하세요.'),
        *messages_for_llm,
    ])

    # add_messages 덕분에 [user_msg, response]만 반환해도 기존 history에 이어 붙음
    return {'chat_history': [user_msg, response]}


multi_workflow = StateGraph(ChatState)
multi_workflow.add_node('chat', chat_node)
# START → chat → END 직접 연결 대신, 아래 셀에서 conditional edge로 분기

In [30]:
from langgraph.checkpoint.memory import MemorySaver

EXIT_KEYWORDS = {'q', 'quit', '종료', '끝'}
CHAT_THREAD_ID = 'multi-chat-session'


def end_chat_router(state: ChatState) -> str:
    """종료 키워드면 END, 아니면 chat 노드로 보냅니다."""
    if state.question.strip().lower() in EXIT_KEYWORDS:
        return END
    return 'chat'


# START에서 종료 여부를 먼저 판단 → chat 또는 END
multi_workflow.add_conditional_edges(START, end_chat_router)
multi_workflow.add_edge('chat', END)

# checkpointer가 thread_id별로 chat_history를 보관 → 셀 재실행해도 이어짐
try:
    chat_memory
except NameError:
    chat_memory = MemorySaver()

multi_app = multi_workflow.compile(checkpointer=chat_memory)
CHAT_CONFIG = {'configurable': {'thread_id': CHAT_THREAD_ID}}

In [32]:
# chat_history를 []로 리셋하지 않음.
# add_messages + checkpointer가 thread에 history를 쌓아 둠 → 새 질문만 넘기면 됨.
print("채팅을 시작합니다. 종료하려면 q / quit / 종료 / 끝 을 입력하세요.")
print("(같은 커널·thread면 셀을 다시 실행해도 이전 대화를 이어갑니다)")

while True:
    user_input = input("\n[User]: ")

    current_state = multi_app.invoke(
        {"question": user_input},
        CHAT_CONFIG,
    )

    if user_input.strip().lower() in EXIT_KEYWORDS:
        print("채팅을 종료합니다.")
        break

    print(f"(history {len(current_state['chat_history'])} msgs)")
    ai_msg = current_state["chat_history"][-1]
    print(f"[AI]: {ai_msg.content}")

채팅을 시작합니다. 종료하려면 q / quit / 종료 / 끝 을 입력하세요.
(같은 커널·thread면 셀을 다시 실행해도 이전 대화를 이어갑니다)
(history 4 msgs)
[AI]: 서린이라는 이름을 가진 친구인데, 너에 대해 더 알고 싶어! 너는 어떤 사람인지, 어떤 취미가 있는지 말해줄 수 있어?
채팅을 종료합니다.


---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'합의'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [ ]:
from typing import Literal


class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0          # optimist가 말할 때마다 +1
    max_turns: int = 3           # optimist 3번 = 총 6메시지 입력 시 종료
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


optimist_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)
skeptic_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)

In [ ]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            '당신은 낙관적인 토론자입니다. 주제에 찬성하는 입장을 말하고,'
            '상대 토론자의 주장이 있다면 반박하세요'
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제: {state.topic}')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = optimist_llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            '당신은 비판적인 토론자입니다. 상대 주장을 반박하세요. 2문장 이내. '
            '더 이상 반박이 어려워 패배를 인정한다면 "합의"라고 말하십시오'
        )),
        *state.chat_history,
    ]
    response = skeptic_llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [ ]:
def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '합의' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

In [ ]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가해 보세요.

* 매 라운드 끝에 양쪽 주장을 한 줄로 요약
* `debate_route`에서 `moderator`를 거친 뒤 `optimist`로 보내기
* 사회자가 '종료'를 언급해야 토론이 종료되도록 종료 조건 수정